# Zonas v2: ¿la zona general falló por el formato o por falta de datos? (Qwen3.5-4B, Colab A100)

**En v1**, la zona general (calibrada con párrafos de WikiText, sin formato de chat) sacó **35 %** en MMLU, contra 72,8 % del modelo completo. Quedó por debajo de la zona de mate (50,5 %). La zona código perdió ~20 puntos en HumanEval (63,4 % contra 82,9 %).

**Qué prueba v2** (rehace solo esas zonas; lo de v1 se reusa desde Drive):

| Versión | Calibración | Tokens | Pregunta |
|---|---|---|---|
| `general_a` | preguntas de opción múltiple en formato chat (MMLU `auxiliary_train`: ARC, OpenBookQA, RACE…; nada del test) | 20k | ¿bastaba con el formato? |
| `general_b` | lo mismo | 80k | ¿además faltaban datos? |
| `code_b` | MBPP en formato chat (todas sus partes) | 80k | ¿a código le faltaban datos? |

Cada una se mide en las 3 tareas (GSM8K 200, HumanEval 164, MMLU 400), igual que en v1. El reporte arma el sistema con la mejor versión de cada zona.

**Uso:** A100 → `Run all` → acepta Drive con **todas las casillas**. ~35–45 min. Si se corta, `Run all` otra vez (lo medido está en `MyDrive/wsi_runs/moi_zones` y se salta). Al final descarga `results_moi_zones_v2.json` y apaga el runtime solo si el resultado está en Drive.

In [ ]:
import os, glob, shutil, json
MODEL   = "Qwen/Qwen3.5-4B"
NEURONS = 6400                                 # de 9216 por capa (69 %), múltiplo de 128
VARIANTS = {"general_a": ("mmlu_aux", 20000), "general_b": ("mmlu_aux", 80000), "code_b": ("mbpp", 80000)}  # versión -> (texto, tokens)
TMP     = "/content/zones"                     # modelos exportados (grandes; no van a Drive)
RES     = "/content/zones/res"                 # resultados (chicos; se copian a Drive)
DATA    = "/content/zones/data"
DRIVE   = "/content/drive/MyDrive/wsi_runs/moi_zones"
OUT     = "results_moi_zones_v2.json"
os.makedirs(RES, exist_ok=True)
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(DRIVE, exist_ok=True)
for f in glob.glob(f"{DRIVE}/*.json"):
    shutil.copy2(f, RES)
def a_drive(nombre):
    if os.path.exists(f"{RES}/{nombre}"):
        shutil.copy2(f"{RES}/{nombre}", DRIVE)
def hecho(nombre):
    return os.path.exists(f"{DRIVE}/{nombre}")
print("ya guardado en Drive:", sorted(os.listdir(DRIVE)) or "nada todavía (empieza de cero)")

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv
!pip install -q vllm
!pip install -q flash-linear-attention || echo "fla opcional"
import vllm, transformers; print("vllm", vllm.__version__, "| transformers", transformers.__version__)

In [ ]:
# Por si vLLM compila piezas de FlashInfer y el enlazador no encuentra libcuda.so (pasó en Kaggle)
import subprocess
real = [l.split("=>")[-1].strip() for l in subprocess.run(["ldconfig", "-p"], capture_output=True, text=True).stdout.splitlines()
        if "libcuda.so" in l and "=>" in l]
real += sorted(glob.glob("/usr/lib64-nvidia/libcuda.so*")) + sorted(glob.glob("/usr/local/nvidia/lib64/libcuda.so*"))
os.makedirs("/tmp/cudalib", exist_ok=True)
if real and not os.path.exists("/tmp/cudalib/libcuda.so"):
    os.symlink(real[0], "/tmp/cudalib/libcuda.so")
os.environ["LIBRARY_PATH"] = "/tmp/cudalib" + (":" + os.environ["LIBRARY_PATH"] if os.environ.get("LIBRARY_PATH") else "")

In [ ]:
%%writefile /content/experiment_moi_speed_vllm.py
"""Does Compensated Selective MoI actually run faster? Real sliced checkpoints served by vLLM.

The LFM3 run (benchmarks/moi_task_index_lfm3) showed the quality side: a compensated math index keeps 79% GSM8K at
70% of MLP neurons. Its speed was measured with HF generate, which is overhead-bound at that size, so no speedup was
visible. vLLM has no LFM3 support yet, so this runs on Qwen3.5-0.8B, which vLLM serves.

  export   calibrate on N GSM8K-train solution tokens (default 20k), pick each layer's top neurons by `energy`
           (sqrt E[h_i^2] * ||W_down[:, i]||, same share in every layer), refit the kept down_proj columns by ridge
           least squares on the co-activation Gram (W_S = W G[:,S] (G_SS + lam I)^-1), and WRITE a real checkpoint
           with smaller gate/up/down matrices and `intermediate_size` = kept neurons. The unused MTP head is sliced
           to the same width (by down_proj column norm) so every shape matches the config.
           --neurons 2560,2048,1792,1024 keeps exact counts instead (multiples of 128: the first T4 run showed
           2509 and 1075 neurons decoding SLOWER than 1792, because partial matmul tiles waste work).
  bench    one checkpoint in vLLM: decode tok/s at batch 1 (the on-device case: weight reads dominate), throughput
           at batch 64, GSM8K accuracy (200 test problems, greedy, thinking off), checkpoint size.
  report   merge the bench files.

    python experiment_moi_speed_vllm.py export --model Qwen/Qwen3.5-0.8B --out-dir /tmp/ckpt --data-dir /tmp/data
    python experiment_moi_speed_vllm.py bench --model-dir /tmp/ckpt/energy_recon_0.5 --tag 0.5 --res-dir R
    python experiment_moi_speed_vllm.py report --res-dir R
"""
from __future__ import annotations

import argparse
import json
import re
import shutil
import statistics
import time
from pathlib import Path

HUB_FILES = {
    "gsm8k/test.parquet": ("openai/gsm8k", "main/test-00000-of-00001.parquet"),
    "gsm8k/train.parquet": ("openai/gsm8k", "main/train-00000-of-00001.parquet"),
    "mbpp/train.parquet": ("google-research-datasets/mbpp", "full/train-00000-of-00001.parquet"),
    "mbpp/validation.parquet": ("google-research-datasets/mbpp", "full/validation-00000-of-00001.parquet"),
    "mbpp/test.parquet": ("google-research-datasets/mbpp", "full/test-00000-of-00001.parquet"),
    "mbpp/prompt.parquet": ("google-research-datasets/mbpp", "full/prompt-00000-of-00001.parquet"),
    "mmlu/auxiliary_train.parquet": ("cais/mmlu", "all/auxiliary_train-00000-of-00001.parquet"),
    "wikitext/train.parquet": ("Salesforce/wikitext", "wikitext-2-raw-v1/train-00000-of-00001.parquet"),
    "humaneval/test.parquet": ("openai/openai_humaneval", "openai_humaneval/test-00000-of-00001.parquet"),
    "mmlu/test.parquet": ("cais/mmlu", "all/test-00000-of-00001.parquet"),
    "mmlu/validation.parquet": ("cais/mmlu", "all/validation-00000-of-00001.parquet"),
}
PROMPT_SUFFIX = "\n\nSolve step by step. End with: The answer is <number>."


def need(data_dir: Path, name: str) -> Path:
    path = data_dir / name
    if path.exists():
        return path
    from huggingface_hub import hf_hub_download

    repo, fn = HUB_FILES[name]
    got = Path(hf_hub_download(repo, fn, repo_type="dataset", local_dir=data_dir / "_hub"))
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_bytes(got.read_bytes())
    return path


def clean_solution(ans: str) -> str:
    body, final = ans.split("####")
    return re.sub(r"<<[^>]*>>", "", body).strip() + f"\nThe answer is {final.strip()}."


def extract_number(text: str) -> str | None:
    text = text.replace(",", "")
    for pat in (r"\\boxed\{([^}]*)\}", r"answer is[:\s]*\$?\s*(-?\d+(?:\.\d+)?)", r"####\s*(-?\d+(?:\.\d+)?)"):
        m = re.findall(pat, text, flags=re.IGNORECASE)
        if m:
            nums = re.findall(r"-?\d+(?:\.\d+)?", m[-1])
            if nums:
                return nums[-1]
    nums = re.findall(r"-?\d+(?:\.\d+)?", text)
    return nums[-1] if nums else None


def same_number(a: str | None, b: str) -> bool:
    try:
        return a is not None and abs(float(a) - float(b)) < 1e-6
    except ValueError:
        return False


def calib_sequences(source: str, tok, data_dir: Path, n_tokens: int) -> list[list[int]]:
    """Token sequences for calibrating a task index ('zone'), until n_tokens are reached.
    gsm8k: GSM8K-train question + worked solution (chat format). mbpp: MBPP-train task + reference Python solution
    (chat format; train first, then the validation / test / prompt splits when more tokens are asked for: the code zone
    is evaluated on HumanEval, a different dataset). wikitext: plain WikiText-2 train paragraphs (general text, no chat
    template). mmlu_aux: MMLU auxiliary_train (ARC, OpenBookQA, RACE... in MMLU's multiple-choice format, none of it
    from the MMLU test), shuffled, asked and answered in chat format exactly like the MMLU evaluation."""
    import pandas as pd
    import numpy as np

    def chat(u, a_):
        return tok(tok.apply_chat_template([{"role": "user", "content": u}, {"role": "assistant", "content": a_}],
                                           tokenize=False, enable_thinking=False), add_special_tokens=False).input_ids

    if source == "gsm8k":
        d = pd.read_parquet(need(data_dir, "gsm8k/train.parquet"))
        gen = (chat(q, clean_solution(ans)) for q, ans in zip(d["question"], d["answer"]))
    elif source == "mbpp":
        d = pd.concat([pd.read_parquet(need(data_dir, f"mbpp/{s}.parquet")) for s in ("train", "validation", "test", "prompt")],
                      ignore_index=True)
        gen = (chat(f"{t}\nYour code should pass this test: {tl[0]}",
                    "```python\n" + c.replace("\r\n", "\n").replace("\t", "    ").strip() + "\n```")
               for t, c, tl in zip(d["text"], d["code"], d["test_list"]))
    elif source == "wikitext":
        d = pd.read_parquet(need(data_dir, "wikitext/train.parquet"))
        paras = [t.strip() for t in d["text"] if len(t.strip()) > 200 and not t.strip().startswith("=")]
        gen = (tok(p_, add_special_tokens=False).input_ids[:1024] for p_ in paras)
    elif source == "mmlu_aux":
        d = pd.read_parquet(need(data_dir, "mmlu/auxiliary_train.parquet"))
        d = d[d["question"].str.len() < 1000]  # mostly short science / commonsense questions, some reading passages
        d = d.iloc[np.random.default_rng(0).permutation(len(d))]
        gen = (chat(q.strip() + "\n" + "\n".join(f"{L}. {c}" for L, c in zip("ABCD", ch)) + "\nAnswer with only the letter.",
                    "ABCD"[int(ans)])
               for q, ch, ans in zip(d["question"], d["choices"], d["answer"]))
    else:
        raise SystemExit(f"unknown calibration source {source}")
    out, n = [], 0
    for ids in gen:
        out.append(ids)
        n += len(ids)
        if n >= n_tokens:
            break
    return out


def local_dir(model: str) -> Path:
    p = Path(model)
    if p.is_dir():
        return p
    from huggingface_hub import snapshot_download

    return Path(snapshot_download(model))


# ---------------------------------------------------------------------------------------------------------- export
def cmd_export(a) -> None:
    import pandas as pd
    import torch
    from safetensors import safe_open
    from safetensors.torch import save_file
    from transformers import AutoModelForCausalLM, AutoTokenizer

    src = local_dir(a.model)
    out = Path(a.out_dir)
    out.mkdir(parents=True, exist_ok=True)
    dev = torch.device(a.device)
    tok = AutoTokenizer.from_pretrained(src)
    def sync() -> None:
        if dev.type == "cuda":
            torch.cuda.synchronize()

    timing: dict = {}
    t_load = time.perf_counter()
    # bf16 where the GPU supports it (the checkpoint's own dtype); fp32 otherwise. Never fp16: these models have large
    # activations that overflow it. Statistics are always accumulated in fp32/fp64.
    cdt = torch.bfloat16 if (dev.type == "cuda" and torch.cuda.get_device_capability()[0] >= 8) else torch.float32
    if a.calib_dtype != "auto":
        cdt = {"bf16": torch.bfloat16, "fp32": torch.float32}[a.calib_dtype]
    model = AutoModelForCausalLM.from_pretrained(src, dtype=cdt).to(dev).eval()
    sync()
    timing["load_model_s"] = round(time.perf_counter() - t_load, 2)
    mods = dict(model.named_modules())
    downs = {}
    for name in mods:
        m = re.search(r"layers\.(\d+)\.mlp\.down_proj$", name)
        if m and "mtp" not in name:
            downs[int(m.group(1))] = name
    L = len(downs)
    U = mods[downs[0]].weight.shape[1]
    print(f"{L} MLPs x {U} neurons", flush=True)

    calib = calib_sequences(a.calib_source, tok, Path(a.data_dir), a.calib_tokens)

    # The Gram matrices of every layer may not fit next to the model (Qwen3.5-4B: 32 x 9216^2 fp32 = 10.9 GB), so the
    # layers are calibrated in groups that fit in free GPU memory; each finished group's Grams go to CPU RAM.
    order = sorted(downs)
    if a.layers_per_pass:
        per = a.layers_per_pass
    elif dev.type == "cuda":
        per = int((torch.cuda.mem_get_info()[0] - 6 * 2**30) // (U * U * 4))
    else:
        per = L
    per = max(1, min(per, L))
    groups = [order[i : i + per] for i in range(0, L, per)]
    Gcpu, sqd = {}, {}
    t0, ntok = time.perf_counter(), 0
    for gi, grp in enumerate(groups):
        st = {li: {"sq": torch.zeros(U, device=dev, dtype=torch.float64),
                   "G": torch.zeros(U, U, device=dev, dtype=torch.float32)} for li in grp}

        def hook(li):
            def f(mod, inp):
                x = inp[0].detach().reshape(-1, U).float()
                st[li]["sq"] += (x * x).sum(0).double()
                st[li]["G"] += x.T @ x
            return f

        hs = [mods[downs[li]].register_forward_pre_hook(hook(li)) for li in grp]
        n_pass = 0
        with torch.no_grad():
            for s in calib:
                model(input_ids=torch.tensor([s], device=dev), use_cache=False)
                n_pass += len(s)
        for h in hs:
            h.remove()
        ntok = n_pass
        for li in grp:
            Gcpu[li], sqd[li] = st[li]["G"].cpu(), st[li]["sq"].cpu()
        del st
        if dev.type == "cuda":
            torch.cuda.empty_cache()
        print(f"  calibration pass {gi + 1}/{len(groups)}: layers {grp[0]}-{grp[-1]}", flush=True)
    sync()
    timing["calibration_s"] = round(time.perf_counter() - t0, 2)
    timing["calibration_passes"] = len(groups)
    print(f"[time] load model {timing['load_model_s']:.1f}s | calibration: {ntok:,} tokens, {len(groups)} pass(es) in "
          f"{timing['calibration_s']:.1f}s", flush=True)

    # The refit only needs each layer's down_proj weight: keep those on CPU and free the whole model from the GPU, so
    # the fp64 solves have room (Qwen3.5-4B: the 9.3 GB model left too little memory and the refit ran out of it).
    import gc

    downW = {li: mods[d].weight.detach().to("cpu") for li, d in downs.items()}
    del model, mods, hs
    gc.collect()
    if dev.type == "cuda":
        torch.cuda.empty_cache()

    files = sorted(src.glob("*.safetensors"))
    keys = {}
    for f in files:
        with safe_open(str(f), "pt") as s:
            for k in s.keys():
                keys[k] = f

    def ckey(li: int, proj: str, mtp: bool = False) -> str:
        pre = "mtp." if mtp else ""
        cands = [k for k in keys if k.endswith(f"layers.{li}.mlp.{proj}.weight") and k.startswith("mtp.") == mtp]
        if len(cands) != 1:
            raise SystemExit(f"cannot find a unique {pre}layer {li} {proj} key: {cands}")
        return cands[0]

    mtp_layers = sorted({int(m.group(1)) for k in keys for m in [re.match(r"mtp\.layers\.(\d+)\.mlp\.down_proj", k)] if m})
    summary = {"model": a.model, "calib_source": a.calib_source, "calib_tokens": ntok, "calib_dtype": str(cdt), "layers": L, "neurons_per_layer": U, "ridge": a.ridge,
               "budgets": {}}
    if a.neurons:  # exact kept-neuron counts, e.g. multiples of 128 so the GPU matmul tiles are full
        targets = [(f"k{int(x)}", int(x)) for x in a.neurons.split(",")]
    else:
        targets = [(f"{float(x):g}", round(float(x) * U)) for x in a.budgets.split(",")]
    for name, k in targets:
        t_rec, t_slice = time.perf_counter(), 0.0
        new, kept = {}, {}
        for li, d in downs.items():
            W = downW[li].to(dev).double()                                # [H, U]
            sq, G = sqd[li].to(dev) / ntok, Gcpu[li].to(dev).double() / ntok
            ix = (sq.sqrt() * W.norm(dim=0)).topk(k).indices.sort().values
            Gss = G[ix][:, ix]
            lam = a.ridge * Gss.diagonal().mean().clamp_min(1e-12)
            Ws = torch.linalg.solve(Gss + lam * torch.eye(k, device=dev, dtype=torch.float64), (W @ G[:, ix]).T).T
            sync()
            t_s = time.perf_counter()
            ixc = ix.cpu()
            kept[li] = ixc.numpy()
            with safe_open(str(keys[ckey(li, "gate_proj")]), "pt") as s:
                gate = s.get_tensor(ckey(li, "gate_proj"))
            with safe_open(str(keys[ckey(li, "up_proj")]), "pt") as s:
                up = s.get_tensor(ckey(li, "up_proj"))
            dt = gate.dtype
            new[ckey(li, "gate_proj")] = gate[ixc].contiguous()
            new[ckey(li, "up_proj")] = up[ixc].contiguous()
            new[ckey(li, "down_proj")] = Ws.to("cpu", dt).contiguous()
            t_slice += time.perf_counter() - t_s
        for li in mtp_layers:  # never used for plain decoding; slice so the shapes match intermediate_size
            with safe_open(str(keys[ckey(li, "down_proj", True)]), "pt") as s:
                dn = s.get_tensor(ckey(li, "down_proj", True))
            ix = dn.float().norm(dim=0).topk(k).indices.sort().values
            new[ckey(li, "down_proj", True)] = dn[:, ix].contiguous()
            for p in ("gate_proj", "up_proj"):
                with safe_open(str(keys[ckey(li, p, True)]), "pt") as s:
                    new[ckey(li, p, True)] = s.get_tensor(ckey(li, p, True))[ix].contiguous()

        recon_s = time.perf_counter() - t_rec - t_slice  # selection + ridge refit only (reading weights excluded)
        t_w = time.perf_counter()
        dst = out / f"energy_recon_{name}"
        if dst.exists():
            shutil.rmtree(dst)
        dst.mkdir(parents=True)
        for f in src.iterdir():
            if f.is_file() and f.suffix != ".safetensors" and f.name != "config.json":
                shutil.copy2(f, dst / f.name)
        cfg = json.loads((src / "config.json").read_text())
        for c in (cfg, cfg.get("text_config", {})):
            if "intermediate_size" in c:
                c["intermediate_size"] = k
        (dst / "config.json").write_text(json.dumps(cfg, indent=2))
        import numpy as np

        np.savez(dst / "kept_neurons.npz", **{f"layer{li}": v for li, v in kept.items()})  # which neurons this index keeps
        for f in files:
            with safe_open(str(f), "pt") as s:
                meta = s.metadata() or {}
                tens = {kk: new[kk] if kk in new else s.get_tensor(kk) for kk in s.keys()}
            save_file(tens, str(dst / f.name), metadata={**meta, "format": "pt"})
            del tens
        write_s = time.perf_counter() - t_w + t_slice
        summary["budgets"][name] = {"kept_neurons": k, "dir": str(dst),
                                    "time_s": {"select_and_compensate": round(recon_s, 2),
                                               "slice_and_write_checkpoint": round(write_s, 2)},
                                    "bytes": sum(f.stat().st_size for f in dst.glob("*.safetensors"))}
        print(f"[time] {name}: select+compensate {recon_s:.1f}s | slice+write {write_s:.1f}s")
        print(f"wrote {dst}  ({k}/{U} neurons per layer, {summary['budgets'][name]['bytes'] / 2**30:.2f} GiB)",
              flush=True)
    summary["timing"] = timing
    summary["device"] = torch.cuda.get_device_name() if dev.type == "cuda" else "cpu"
    summary["full_bytes"] = sum(f.stat().st_size for f in files)
    summary["full_dir"] = str(src)
    (out / "export.json").write_text(json.dumps(summary, indent=2))


# ----------------------------------------------------------------------------------------------------------- bench
def cmd_bench(a) -> None:
    import pandas as pd
    import torch
    from transformers import AutoTokenizer
    from vllm import LLM, SamplingParams

    md = local_dir(a.model_dir)
    tok = AutoTokenizer.from_pretrained(md)
    dtype = a.dtype
    if dtype == "auto" and torch.cuda.get_device_capability()[0] < 8:
        dtype = "half"  # T4 / P100: no bf16
    kw = dict(model=str(md), dtype=dtype, max_model_len=a.max_model_len, gpu_memory_utilization=a.gpu_mem, seed=0)
    llm = None
    for extra in ({"limit_mm_per_prompt": {"image": 0, "video": 0}}, {}):
        try:
            llm = LLM(**kw, **extra)
            break
        except Exception as e:  # noqa: BLE001
            print("vLLM start failed with", extra, "->", repr(e)[:300], flush=True)
    if llm is None:
        raise SystemExit("vLLM could not start")

    test = pd.read_parquet(need(Path(a.data_dir), "gsm8k/test.parquet")).iloc[: a.n_problems]
    prompts = [tok.apply_chat_template([{"role": "user", "content": q + PROMPT_SUFFIX}], tokenize=False,
                                       add_generation_prompt=True, enable_thinking=False) for q in test["question"]]

    def timed(ps: list[str], n_new: int) -> float:
        sp = SamplingParams(temperature=0.0, max_tokens=n_new, ignore_eos=True)
        t = time.perf_counter()
        outs = llm.generate(ps, sp, use_tqdm=False)
        dt = time.perf_counter() - t
        return sum(len(o.outputs[0].token_ids) for o in outs) / dt

    timed(prompts[:4], 32)  # warmup (CUDA graphs, kernels)
    b1 = [timed(prompts[i : i + 1], a.decode_tokens) for i in range(a.repeats)]
    thr = [timed(prompts[:a.batch], a.batch_tokens) for _ in range(2)]
    print(f"[{a.tag}] batch 1: {statistics.median(b1):.1f} tok/s   batch {a.batch}: {max(thr):.0f} tok/s", flush=True)

    t = time.perf_counter()
    outs = llm.generate(prompts, SamplingParams(temperature=0.0, max_tokens=a.max_new), use_tqdm=False)
    gen_s = time.perf_counter() - t
    ok, samples, ntoks = 0, [], 0
    for o, gold in zip(outs, test["answer"]):
        g = gold.split("####")[-1].strip().replace(",", "")
        txt = o.outputs[0].text
        pred = extract_number(txt.split("</think>")[-1])
        ok += same_number(pred, g)
        ntoks += len(o.outputs[0].token_ids)
        if len(samples) < 3:
            samples.append({"gold": g, "pred": pred, "text": txt[-300:]})
    sizes = sum(f.stat().st_size for f in md.glob("*.safetensors"))
    cfg = json.loads((md / "config.json").read_text())
    res = {"tag": a.tag, "model_dir": str(md), "dtype": dtype, "gpu": torch.cuda.get_device_name(),
           "intermediate_size": cfg.get("text_config", cfg).get("intermediate_size"),
           "checkpoint_bytes": sizes,
           "decode_tok_s_batch1": round(statistics.median(b1), 2), "decode_tok_s_batch1_runs": [round(x, 2) for x in b1],
           f"throughput_tok_s_batch{a.batch}": round(max(thr), 1),
           "gsm8k": {"accuracy": ok / len(prompts), "correct": ok, "n": len(prompts), "gen_tokens": ntoks,
                     "wall_s": round(gen_s, 1), "samples": samples}}
    rd = Path(a.res_dir)
    rd.mkdir(parents=True, exist_ok=True)
    (rd / f"bench_{a.tag}.json").write_text(json.dumps(res, indent=2))
    print(f"[{a.tag}] GSM8K {ok}/{len(prompts)} = {ok / len(prompts):.1%}  ({gen_s:.0f}s)", flush=True)


def cmd_report(a) -> None:
    rd = Path(a.res_dir)
    rows = {}
    for f in sorted(rd.glob("bench_*.json")):
        r = json.loads(f.read_text())
        rows[r["tag"]] = r
    exp = Path(a.export_json)
    res = {"export": json.loads(exp.read_text()) if exp.exists() else None, "arms": rows}
    if res["export"] and "timing" in res["export"]:
        t = res["export"]["timing"]
        print(f"\nBuild time on {res['export'].get('device')}: load {t['load_model_s']:.1f}s, calibration "
              f"({res['export']['calib_tokens']:,} tokens) {t['calibration_s']:.1f}s")
        for n, bgt in res["export"]["budgets"].items():
            ts = bgt.get("time_s", {})
            print(f"  {n}: select+compensate {ts.get('select_and_compensate', 0):.1f}s, "
                  f"slice+write {ts.get('slice_and_write_checkpoint', 0):.1f}s")
    if "full" in rows:
        f = rows["full"]
        bk = next(k for k in f if k.startswith("throughput_tok_s_batch"))
        print(f"\n{'arm':6s} {'neurons':>8s} {'size':>8s} {'GSM8K':>7s} {'batch1':>12s} {'batch-N':>12s}")
        for t, r in rows.items():
            print(f"{t:6s} {r['intermediate_size']:8d} {r['checkpoint_bytes'] / 2**30:7.2f}G "
                  f"{r['gsm8k']['accuracy']:7.1%} {r['decode_tok_s_batch1']:7.1f} ({r['decode_tok_s_batch1'] / f['decode_tok_s_batch1']:.2f}x)"
                  f" {r[bk]:7.0f} ({r[bk] / f[bk]:.2f}x)")
        res["speedup_vs_full"] = {t: {"batch1": round(r["decode_tok_s_batch1"] / f["decode_tok_s_batch1"], 3),
                                      "batchN": round(r[bk] / f[bk], 3),
                                      "size": round(r["checkpoint_bytes"] / f["checkpoint_bytes"], 3)}
                                  for t, r in rows.items()}
    Path(a.out).write_text(json.dumps(res, indent=2))
    print("Saved", a.out)


def main() -> None:
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    sub = ap.add_subparsers(dest="cmd", required=True)
    e = sub.add_parser("export")
    e.add_argument("--model", default="Qwen/Qwen3.5-0.8B")
    e.add_argument("--out-dir", required=True)
    e.add_argument("--data-dir", required=True)
    e.add_argument("--budgets", default="0.7,0.5,0.3")
    e.add_argument("--neurons", help="exact kept neurons per layer, comma-separated (overrides --budgets)")
    e.add_argument("--calib-tokens", type=int, default=20_000)
    e.add_argument("--calib-source", default="gsm8k", choices=["gsm8k", "mbpp", "wikitext", "mmlu_aux"],
                   help="text the index is calibrated on (one 'zone' per task)")
    e.add_argument("--ridge", type=float, default=1e-2)
    e.add_argument("--device", default="cuda")
    e.add_argument("--calib-dtype", default="auto", choices=["auto", "bf16", "fp32"])
    e.add_argument("--layers-per-pass", type=int, default=0, help="0 = as many as fit in free GPU memory")
    b = sub.add_parser("bench")
    b.add_argument("--model-dir", required=True)
    b.add_argument("--tag", required=True)
    b.add_argument("--res-dir", required=True)
    b.add_argument("--data-dir", required=True)
    b.add_argument("--n-problems", type=int, default=200)
    b.add_argument("--max-new", type=int, default=1024)
    b.add_argument("--decode-tokens", type=int, default=512)
    b.add_argument("--repeats", type=int, default=5)
    b.add_argument("--batch", type=int, default=64)
    b.add_argument("--batch-tokens", type=int, default=256)
    b.add_argument("--dtype", default="auto")
    b.add_argument("--max-model-len", type=int, default=4096)
    b.add_argument("--gpu-mem", type=float, default=0.85)
    r = sub.add_parser("report")
    r.add_argument("--res-dir", required=True)
    r.add_argument("--export-json", required=True)
    r.add_argument("--out", required=True)
    a = ap.parse_args()
    {"export": cmd_export, "bench": cmd_bench, "report": cmd_report}[a.cmd](a)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/moi_zones.py
"""Zone streaming demo for Compensated Selective MoI: one model, several task zones, a router, and the cost of a swap.

Like a game that only loads the part of the map in view: each "zone" is a compensated task index (about 70% of the MLP
neurons, built from ~20k tokens of that task) and only the active zone needs to be in GPU memory. A router reads the
prompt BEFORE generation and picks the zone. The rest of the model (attention, embeddings, norms) stays resident.

Zones (built with experiment_moi_speed_vllm.py export --calib-source ...):
    math     GSM8K-train worked solutions          -> evaluated on GSM8K test (200)
    code     MBPP-train tasks + reference code     -> evaluated on HumanEval (164, run against its unit tests)
    general  WikiText-2 paragraphs                  -> evaluated on MMLU test (400, letter answer)
Calibration and evaluation sets never overlap; code is calibrated on MBPP and tested on HumanEval (different dataset).

Subcommands:
    eval    one checkpoint (full model or a zone) in vLLM on all three tasks; per-item correctness is saved, so every
            zone x task cell and any routing policy can be scored afterwards. Greedy, thinking off.
    route   the router: nearest centroid on the mean INPUT EMBEDDING of the prompt (no forward pass; the embedding
            table stays resident anyway). Trained on GSM8K-train questions, MBPP-train tasks and MMLU-validation
            questions; tested on the eval prompts' content, without the task instructions (no trivial cue).
    swap    the cost of switching zones: bytes of a zone's MLP weights; load time disk -> GPU (cold page cache if it
            can be dropped, and warm), pinned RAM -> GPU; network time estimated from bytes; neuron overlap between
            zones and the bytes a delta swap would move.
    report  zone x task accuracy, routed system vs full model, cost of a wrong zone, routing accuracy, swap costs.
"""
from __future__ import annotations

import argparse
import json
import os
import re
import subprocess
import sys
import tempfile
import time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np

sys.path.insert(0, str(Path(__file__).resolve().parent))
import experiment_moi_speed_vllm as sp  # noqa: E402

ZONES = {"math": "gsm8k", "code": "mbpp", "general": "wikitext"}  # zone -> calibration source
TASKS = ["gsm8k", "humaneval", "mmlu"]
TASK_ZONE = {"gsm8k": "math", "humaneval": "code", "mmlu": "general"}
LET = "ABCD"
CODE_INSTR = ("Complete the following Python function. Reply with the complete function, including any imports, "
              "in a single ```python code block.\n\n")
MC_INSTR = "\nAnswer with only the letter."
MAX_NEW = {"gsm8k": 1024, "humaneval": 768, "mmlu": 8}


# ------------------------------------------------------------------ tasks
def mc_text(q, choices):
    return q.strip() + "\n" + "\n".join(f"{L}. {c}" for L, c in zip(LET, choices))


def load_tasks(data_dir: Path, n_gsm: int, n_mmlu: int, seed: int) -> dict:
    """task -> list of items with 'content' (what the router sees) and 'user' (the full message sent to the model)."""
    import pandas as pd

    out = {}
    g = pd.read_parquet(sp.need(data_dir, "gsm8k/test.parquet")).iloc[:n_gsm]
    out["gsm8k"] = [{"content": q, "user": q + sp.PROMPT_SUFFIX, "gold": a.split("####")[-1].strip().replace(",", "")}
                    for q, a in zip(g["question"], g["answer"])]
    h = pd.read_parquet(sp.need(data_dir, "humaneval/test.parquet"))
    out["humaneval"] = [{"content": p, "user": CODE_INSTR + p, "prompt": p, "test": t, "entry": e, "task_id": i}
                        for i, p, t, e in zip(h["task_id"], h["prompt"], h["test"], h["entry_point"])]
    m = pd.read_parquet(sp.need(data_dir, "mmlu/test.parquet"))
    idx = np.random.default_rng(seed).permutation(len(m))[:n_mmlu]
    out["mmlu"] = [{"content": mc_text(m["question"].iloc[i], list(m["choices"].iloc[i])),
                    "user": mc_text(m["question"].iloc[i], list(m["choices"].iloc[i])) + MC_INSTR,
                    "gold": int(m["answer"].iloc[i]), "subject": m["subject"].iloc[i]} for i in idx]
    return out


def extract_code(text: str, prompt: str, entry: str) -> str:
    blocks = re.findall(r"```(?:python|py|Python)?[ \t]*\n(.*?)```", text, re.S)
    if not blocks and "```" in text:  # unterminated block (hit the token cap)
        blocks = [text.split("```", 1)[1].split("\n", 1)[-1]]
    with_def = [b for b in blocks if f"def {entry}" in b]
    code = (with_def or blocks or [text])[-1]
    if f"def {entry}" not in code:  # the model wrote only the body
        code = prompt + code
    elif not re.search(r"^(import|from) ", code, re.M):
        # keep the prompt's imports (typing etc.) if the model dropped them
        imports = "\n".join(l for l in prompt.splitlines() if re.match(r"^(import|from) ", l))
        code = imports + "\n" + code
    return code


def run_humaneval(code: str, test: str, entry: str, timeout: float = 10.0) -> bool:
    prog = f"{code}\n\n{test}\n\ncheck({entry})\n"
    with tempfile.NamedTemporaryFile("w", suffix=".py", delete=False, encoding="utf-8") as f:
        f.write(prog)
        path = f.name
    try:
        r = subprocess.run([sys.executable, path], capture_output=True, timeout=timeout,
                           cwd=tempfile.gettempdir())
        return r.returncode == 0
    except subprocess.TimeoutExpired:
        return False
    finally:
        os.unlink(path)


def grade_mc(text: str, gold: int) -> bool:
    t = text.strip()
    m = re.match(r"^\(?([ABCD])\b", t) or re.search(r"\b([ABCD])\b", t[:40])
    return bool(m) and LET.index(m.group(1)) == gold


def grade(task: str, item: dict, text: str) -> bool:
    if task == "gsm8k":
        return sp.same_number(sp.extract_number(text), item["gold"])
    if task == "humaneval":
        return run_humaneval(extract_code(text, item["prompt"], item["entry"]), item["test"], item["entry"])
    return grade_mc(text, item["gold"])


# ------------------------------------------------------------------ eval (vLLM)
def cmd_eval(a) -> None:
    import torch
    from transformers import AutoTokenizer
    from vllm import LLM, SamplingParams

    md = sp.local_dir(a.model_dir)
    tok = AutoTokenizer.from_pretrained(md)
    tasks = load_tasks(Path(a.data_dir), a.n_gsm, a.n_mmlu, a.seed)
    dtype = a.dtype
    if dtype == "auto" and torch.cuda.get_device_capability()[0] < 8:
        dtype = "half"
    t_load = time.perf_counter()
    kw = dict(model=str(md), dtype=dtype, max_model_len=a.max_model_len, gpu_memory_utilization=a.gpu_mem, seed=0)
    llm = None
    for extra in ({"limit_mm_per_prompt": {"image": 0, "video": 0}}, {}):
        try:
            llm = LLM(**kw, **extra)
            break
        except Exception as e:  # noqa: BLE001
            print("vLLM start failed with", extra, "->", repr(e)[:300], flush=True)
    if llm is None:
        raise SystemExit("vLLM could not start")
    load_s = time.perf_counter() - t_load

    prompts, params, where = [], [], []
    for task in TASKS:
        for i, it in enumerate(tasks[task]):
            prompts.append(tok.apply_chat_template([{"role": "user", "content": it["user"]}], tokenize=False,
                                                   add_generation_prompt=True, enable_thinking=False))
            params.append(SamplingParams(temperature=0.0, max_tokens=MAX_NEW[task]))
            where.append((task, i))
    t0 = time.perf_counter()
    outs = llm.generate(prompts, params, use_tqdm=False)
    gen_s = time.perf_counter() - t0
    texts = {t: [None] * len(tasks[t]) for t in TASKS}
    ntok = 0
    for (task, i), o in zip(where, outs):
        texts[task][i] = o.outputs[0].text
        ntok += len(o.outputs[0].token_ids)
    t0 = time.perf_counter()
    res = {"tag": a.tag, "model_dir": str(md), "gpu": torch.cuda.get_device_name(0),
           "checkpoint_bytes": sum(f.stat().st_size for f in md.glob("*.safetensors")), "tasks": {}}
    for task in TASKS:
        if task == "humaneval":
            with ThreadPoolExecutor(8) as ex:
                ok = list(ex.map(lambda p: grade(task, *p), zip(tasks[task], texts[task])))
        else:
            ok = [grade(task, it, tx) for it, tx in zip(tasks[task], texts[task])]
        res["tasks"][task] = {"n": len(ok), "accuracy": round(float(np.mean(ok)), 4), "correct": [int(x) for x in ok],
                              "samples": [tx[-400:] for tx in texts[task][:2]]}
        print(f"[{a.tag}] {task}: {np.mean(ok):.1%} ({sum(ok)}/{len(ok)})", flush=True)
    res["time_s"] = {"vllm_load": round(load_s, 1), "generate": round(gen_s, 1),
                     "grade": round(time.perf_counter() - t0, 1), "gen_tokens": ntok}
    Path(a.res_dir).mkdir(parents=True, exist_ok=True)
    (Path(a.res_dir) / f"eval_{a.tag}.json").write_text(json.dumps(res, indent=1))


# ------------------------------------------------------------------ router (input embeddings, no forward pass)
def embed_table(model_dir: Path):
    import torch
    from safetensors import safe_open

    for f in sorted(model_dir.glob("*.safetensors")):
        with safe_open(str(f), "pt") as s:
            for k in s.keys():
                if k.endswith("embed_tokens.weight") and not k.startswith("mtp."):
                    return s.get_tensor(k).to(torch.float32)
    raise SystemExit("no embed_tokens.weight found")


def stub(text: str, code: str, tests) -> str:
    """An MBPP task written as a function signature + docstring with one doctest-style example."""
    m = re.search(r"^def .*?:\s*$", code.replace("\r", ""), re.M)
    sig = m.group(0) if m else "def solution():"
    return f'{sig}\n    """ {text}\n    >>> {tests[0].replace("assert ", "")}\n    """\n'


def cmd_route(a) -> None:
    import pandas as pd
    from transformers import AutoTokenizer

    md = sp.local_dir(a.model)
    tok = AutoTokenizer.from_pretrained(md)
    t0 = time.perf_counter()
    E = embed_table(md)
    load_s = time.perf_counter() - t0
    dd = Path(a.data_dir)

    def feat(texts):
        out = []
        for t in texts:
            ids = tok(t, add_special_tokens=False).input_ids[:512] or [0]
            v = E[ids].mean(0).numpy()
            out.append(v / (np.linalg.norm(v) + 1e-9))
        return np.stack(out)

    n = a.n_train
    g = pd.read_parquet(sp.need(dd, "gsm8k/train.parquet"))
    mb = pd.read_parquet(sp.need(dd, "mbpp/train.parquet"))
    mv = pd.read_parquet(sp.need(dd, "mmlu/validation.parquet"))
    # code requests come both as prose + code and as a function stub with a docstring: half of the MBPP-train tasks
    # are written each way (built from MBPP only; HumanEval is never seen by the router)
    h = n // 2
    train = {"math": list(g["question"][:n]),
             "code": [f"{t}\n{c}" for t, c in zip(mb["text"][:h], mb["code"][:h])]
                     + [stub(t, c, tl) for t, c, tl in zip(mb["text"][h:n], mb["code"][h:n], mb["test_list"][h:n])],
             "general": [mc_text(q, list(c)) for q, c in zip(mv["question"][:n], mv["choices"][:n])]}
    from sklearn.linear_model import LogisticRegression

    F = {z: feat(v) for z, v in train.items()}
    zones = list(F)
    X_tr = np.concatenate([F[z] for z in zones])
    y_tr = np.concatenate([[j] * len(F[z]) for j, z in enumerate(zones)])
    mu, sd = X_tr.mean(0), X_tr.std(0) + 1e-6
    clf = LogisticRegression(max_iter=2000, C=a.reg).fit((X_tr - mu) / sd, y_tr)
    C = np.stack([(F[z] - mu).mean(0) for z in zones])  # nearest-centroid baseline, reported for comparison
    C /= np.linalg.norm(C, axis=1, keepdims=True)
    t1 = time.perf_counter()
    tasks = load_tasks(dd, a.n_gsm, a.n_mmlu, a.seed)
    res = {"method": "logistic regression on the mean input embedding of the prompt content (no forward pass)",
           "train_per_zone": {z: len(v) for z, v in train.items()}, "embed_load_s": round(load_s, 2), "tasks": {}}
    nprompts = 0
    for task in TASKS:
        Xr = feat([it["content"] for it in tasks[task]])
        pred = [zones[j] for j in clf.predict((Xr - mu) / sd)]
        cen = [zones[j] for j in ((Xr - mu) @ C.T).argmax(1)]
        nprompts += len(pred)
        res["tasks"][task] = {"route": pred, "accuracy": round(float(np.mean([p == TASK_ZONE[task] for p in pred])), 4),
                              "counts": {z: pred.count(z) for z in zones},
                              "centroid_accuracy": round(float(np.mean([p == TASK_ZONE[task] for p in cen])), 4)}
        print(f"router on {task}: {res['tasks'][task]['accuracy']:.1%} -> {res['tasks'][task]['counts']} "
              f"(nearest centroid {res['tasks'][task]['centroid_accuracy']:.1%})", flush=True)
    res["route_ms_per_prompt"] = round((time.perf_counter() - t1) / max(nprompts, 1) * 1000, 3)
    Path(a.res_dir).mkdir(parents=True, exist_ok=True)
    (Path(a.res_dir) / "route.json").write_text(json.dumps(res, indent=1))


# ------------------------------------------------------------------ swap cost
def mlp_keys(model_dir: Path) -> dict:
    from safetensors import safe_open

    keys = {}
    for f in sorted(model_dir.glob("*.safetensors")):
        with safe_open(str(f), "pt") as s:
            for k in s.keys():
                if re.search(r"layers\.\d+\.mlp\.(gate|up|down)_proj\.weight$", k) and not k.startswith("mtp."):
                    keys[k] = f
    return keys


def drop_page_cache() -> bool:
    try:
        subprocess.run(["sync"], check=False)
        with open("/proc/sys/vm/drop_caches", "w") as f:
            f.write("3\n")
        return True
    except Exception:  # noqa: BLE001
        return False


def cmd_swap(a) -> None:
    import torch
    from safetensors import safe_open

    dev = "cuda" if torch.cuda.is_available() and not a.cpu else "cpu"

    def sync():
        if dev == "cuda":
            torch.cuda.synchronize()

    zdirs = dict(x.split("=", 1) for x in a.zones.split(","))
    res = {"device": torch.cuda.get_device_name(0) if dev == "cuda" else "cpu", "zones": {}}
    full = sp.local_dir(a.full) if a.full else None
    if full:
        fk = mlp_keys(full)
        total = sum(f.stat().st_size for f in full.glob("*.safetensors"))
        mlp_b = 0
        for k, f in fk.items():
            with safe_open(str(f), "pt") as s:
                t = s.get_slice(k)
                mlp_b += int(np.prod(t.get_shape())) * 2
        res["full"] = {"checkpoint_bytes": total, "mlp_bytes": mlp_b, "resident_non_mlp_bytes": total - mlp_b}
    kept = {}
    for z, d in zdirs.items():
        d = Path(d)
        keys = mlp_keys(d)
        if (d / "kept_neurons.npz").exists():
            kept[z] = dict(np.load(d / "kept_neurons.npz"))
        r = {}
        # disk -> device, cold page cache if we can drop it, then warm
        for label in ("disk_cold", "disk_warm"):
            if label == "disk_cold":
                r["page_cache_dropped"] = drop_page_cache()
            sync()
            t0 = time.perf_counter()
            nbytes = 0
            for k, f in keys.items():
                with safe_open(str(f), "pt", device=dev) as s:
                    t = s.get_tensor(k)
                    nbytes += t.numel() * t.element_size()
            sync()
            r[f"{label}_s"] = round(time.perf_counter() - t0, 3)
        r["mlp_bytes"] = nbytes
        # pinned RAM -> device
        cpu = []
        for k, f in keys.items():
            with safe_open(str(f), "pt") as s:
                t = s.get_tensor(k).clone()  # own the memory: the file mapping closes with the context
            cpu.append(t.pin_memory() if dev == "cuda" else t)
        gpu = [torch.empty_like(t, device=dev) for t in cpu]
        times = []
        for _ in range(3):
            sync()
            t0 = time.perf_counter()
            for g_, c in zip(gpu, cpu):
                g_.copy_(c, non_blocking=True)
            sync()
            times.append(time.perf_counter() - t0)
        r["ram_to_gpu_s"] = round(min(times), 4)
        r["ram_to_gpu_GBps"] = round(nbytes / min(times) / 1e9, 2)
        r["network_estimate_s"] = {"100Mbps": round(nbytes * 8 / 1e8, 1), "1Gbps": round(nbytes * 8 / 1e9, 2)}
        del cpu, gpu
        res["zones"][z] = r
        print(f"zone {z}: {nbytes / 2**30:.2f} GiB of MLP | disk cold {r['disk_cold_s']}s "
              f"(cache dropped: {r['page_cache_dropped']}) | warm {r['disk_warm_s']}s | RAM->GPU {r['ram_to_gpu_s']}s",
              flush=True)
    # overlap between zones and the bytes of a delta swap (new gate/up rows + every down column, which is refit per zone)
    names = list(kept)
    ov = {}
    for i in range(len(names)):
        for j in range(i + 1, len(names)):
            A, B = kept[names[i]], kept[names[j]]
            shared = [len(np.intersect1d(A[k], B[k])) / len(A[k]) for k in A]
            ov[f"{names[i]}-{names[j]}"] = round(float(np.mean(shared)), 4)
    res["kept_neuron_overlap"] = ov
    if names:
        z0 = res["zones"][names[0]]["mlp_bytes"]
        mean_ov = float(np.mean(list(ov.values()))) if ov else 1.0
        res["delta_swap_bytes_estimate"] = int(z0 * (2 / 3) * (1 - mean_ov) + z0 / 3)
    Path(a.res_dir).mkdir(parents=True, exist_ok=True)
    (Path(a.res_dir) / "swap.json").write_text(json.dumps(res, indent=1))


# ------------------------------------------------------------------ report
def cmd_report(a) -> None:
    rd = Path(a.res_dir)
    ev = {p.stem[len("eval_"):]: json.loads(p.read_text()) for p in sorted(rd.glob("eval_*.json"))}
    route = json.loads((rd / "route.json").read_text()) if (rd / "route.json").exists() else None
    swap = json.loads((rd / "swap.json").read_text()) if (rd / "swap.json").exists() else None
    exports = {}
    for p in sorted(rd.glob("export_*.json")):
        exports[p.stem[len("export_"):]] = json.loads(p.read_text())
    acc = {tag: {t: e["tasks"][t]["accuracy"] for t in TASKS} for tag, e in ev.items()}
    res = {"accuracy": acc, "checkpoint_bytes": {tag: e["checkpoint_bytes"] for tag, e in ev.items()}}
    zones = [z for z in ZONES if z in ev]
    if "full" in ev and zones:
        sysr = {}
        for t in TASKS:
            n = ev["full"]["tasks"][t]["n"]
            own = TASK_ZONE[t]
            row = {"full": acc["full"][t]}
            if own in ev:
                row["own_zone"] = acc[own][t]
            others = [acc[z][t] for z in zones if z != own]
            if others:
                row["wrong_zone_mean"] = round(float(np.mean(others)), 4)
                row["wrong_zone_worst"] = min(others)
            if route and all(z in ev for z in set(route["tasks"][t]["route"])):
                picks = route["tasks"][t]["route"]
                row["routed"] = round(float(np.mean([ev[z]["tasks"][t]["correct"][i] for i, z in enumerate(picks)])), 4)
                row["router_accuracy"] = route["tasks"][t]["accuracy"]
            row["n"] = n
            sysr[t] = row
        res["system"] = sysr
        res["system_mean"] = {k: round(float(np.mean([sysr[t][k] for t in TASKS if k in sysr[t]])), 4)
                              for k in ("full", "own_zone", "routed", "wrong_zone_mean") if all(k in sysr[t] for t in TASKS)}
    # zone variants (e.g. general_a = another calibration of the general zone): each one against its zone's own task,
    # and the system rebuilt with the best variant of every zone. That pick looks at the test, so it is a best case.
    variants = {}
    for tag in acc:
        z = tag.split("_")[0]
        if z in ZONES:
            own_t = next(t for t, zz in TASK_ZONE.items() if zz == z)
            ex = exports.get(tag, {})
            variants[tag] = {"zone": z, "own_task": own_t, "own_task_acc": acc[tag][own_t],
                             "calib_source": ex.get("calib_source"), "calib_tokens": ex.get("calib_tokens")}
    if any(tag not in ZONES for tag in variants):
        res["variants"] = variants
        best = {z: max((t for t, v in variants.items() if v["zone"] == z), key=lambda t: variants[t]["own_task_acc"])
                for z in ZONES if any(v["zone"] == z for v in variants.values())}
        if "full" in ev and route and len(best) == len(ZONES):
            sb = {}
            for t in TASKS:
                picks = route["tasks"][t]["route"]
                sb[t] = {"own_zone": acc[best[TASK_ZONE[t]]][t],
                         "routed": round(float(np.mean([ev[best[z]]["tasks"][t]["correct"][i] for i, z in enumerate(picks)])), 4)}
            res["system_best_variants"] = {"zones": best, "tasks": sb, "selected_on_test": True,
                                           "mean_routed": round(float(np.mean([sb[t]["routed"] for t in TASKS])), 4),
                                           "mean_full": res.get("system_mean", {}).get("full")}
    if route:
        res["router"] = {k: v for k, v in route.items() if k != "tasks"}
        res["router"]["accuracy"] = {t: route["tasks"][t]["accuracy"] for t in TASKS}
        res["router"]["counts"] = {t: route["tasks"][t]["counts"] for t in TASKS}
    if swap:
        res["swap"] = swap
    if exports:
        res["build"] = {z: {"calib_tokens": e.get("calib_tokens"), "timing": e.get("timing"),
                            "budgets": {k: v.get("time_s") for k, v in e.get("budgets", {}).items()}}
                        for z, e in exports.items()}
    res["eval_time_s"] = {tag: e.get("time_s") for tag, e in ev.items()}
    Path(a.out).write_text(json.dumps(res, indent=1))

    print(f"\n{'':10s}" + "".join(f"{t:>11s}" for t in TASKS) + "   checkpoint")
    for tag in ["full"] + zones + sorted(t for t in acc if t != "full" and t not in zones):
        if tag in acc:
            print(f"{tag:10s}" + "".join(f"{acc[tag][t]:11.1%}" for t in TASKS)
                  + f"   {res['checkpoint_bytes'][tag] / 2**30:.2f} GiB")
    if "system" in res:
        print("\nper task: full | own zone | routed | wrong zone (mean / worst) | router accuracy")
        for t, r in res["system"].items():
            print(f"  {t:10s} {r['full']:.1%} | {r.get('own_zone', float('nan')):.1%} | {r.get('routed', float('nan')):.1%} | "
                  f"{r.get('wrong_zone_mean', float('nan')):.1%} / {r.get('wrong_zone_worst', float('nan')):.1%} | "
                  f"{r.get('router_accuracy', float('nan')):.1%}")
        print("mean over tasks:", res["system_mean"])
    if "variants" in res:
        print("\nzone variants (own task):")
        for tag, v in res["variants"].items():
            print(f"  {tag:12s} {v['own_task']:10s} {v['own_task_acc']:.1%}  calib {v['calib_source']} {v['calib_tokens']}")
        if "system_best_variants" in res:
            print("best variant per zone (picked on the test):", res["system_best_variants"])
    if swap:
        for z, r in swap["zones"].items():
            print(f"swap {z}: {r['mlp_bytes'] / 2**30:.2f} GiB | disk cold {r['disk_cold_s']}s | warm {r['disk_warm_s']}s | "
                  f"RAM->GPU {r['ram_to_gpu_s']}s ({r['ram_to_gpu_GBps']} GB/s) | 1 Gbps ~{r['network_estimate_s']['1Gbps']}s")
        print("kept-neuron overlap:", swap.get("kept_neuron_overlap"))
    print("saved", a.out)


def main() -> None:
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    sub = ap.add_subparsers(dest="cmd", required=True)
    for name in ("eval", "route"):
        p = sub.add_parser(name)
        p.add_argument("--data-dir", required=True)
        p.add_argument("--res-dir", required=True)
        p.add_argument("--n-gsm", type=int, default=200)
        p.add_argument("--n-mmlu", type=int, default=400)
        p.add_argument("--seed", type=int, default=0)
        if name == "eval":
            p.add_argument("--model-dir", required=True)
            p.add_argument("--tag", required=True)
            p.add_argument("--dtype", default="auto")
            p.add_argument("--max-model-len", type=int, default=4096)
            p.add_argument("--gpu-mem", type=float, default=0.85)
        else:
            p.add_argument("--model", required=True)
            p.add_argument("--n-train", type=int, default=300)
            p.add_argument("--reg", type=float, default=0.1, help="inverse L2 strength of the router's logistic regression")
    s = sub.add_parser("swap")
    s.add_argument("--zones", required=True, help="math=DIR,code=DIR,general=DIR")
    s.add_argument("--full", help="full model (hub id or dir), for the resident / MLP byte split")
    s.add_argument("--res-dir", required=True)
    s.add_argument("--cpu", action="store_true")
    r = sub.add_parser("report")
    r.add_argument("--res-dir", required=True)
    r.add_argument("--out", required=True)
    a = ap.parse_args()
    {"eval": cmd_eval, "route": cmd_route, "swap": cmd_swap, "report": cmd_report}[a.cmd](a)


if __name__ == "__main__":
    main()

In [ ]:
# 1. construir las versiones nuevas de las zonas (calibrar, elegir neuronas, compensar, escribir el modelo)
for z, (src, ntok) in VARIANTS.items():
    d = f"{TMP}/ckpt_{z}/energy_recon_k{NEURONS}"
    if hecho(f"eval_{z}.json") or os.path.exists(d):
        continue
    print(f"========== {z} (calibrada con {src}, {ntok:,} tokens) ==========", flush=True)
    !python -u /content/experiment_moi_speed_vllm.py export --model {MODEL} --out-dir {TMP}/ckpt_{z} --data-dir {DATA} --neurons {NEURONS} --calib-source {src} --calib-tokens {ntok} 2>&1 | grep -v -E "Loading weights|fast path|Warning"
    shutil.copy2(f"{TMP}/ckpt_{z}/export.json", f"{RES}/export_{z}.json")
    a_drive(f"export_{z}.json")

In [ ]:
# 2. medir cada versión en las 3 tareas (cada una en su propio proceso de vLLM)
for z in VARIANTS:
    if hecho(f"eval_{z}.json"):
        print(f"{z}: ya medido (Drive)"); continue
    d = f"{TMP}/ckpt_{z}/energy_recon_k{NEURONS}"
    print(f"========== {z} ==========", flush=True)
    !python -u /content/moi_zones.py eval --model-dir {d} --tag {z} --data-dir {DATA} --res-dir {RES} 2>&1 | grep --line-buffered -v -E "^(INFO|DEBUG)|it/s\]|Attempted to load|Warning"
    a_drive(f"eval_{z}.json")

In [ ]:
# 3. lo de v1 (modelo completo, zonas v1, router) tiene que estar en Drive para el reporte
faltan = [f for f in ["eval_full.json", "eval_math.json", "eval_code.json", "eval_general.json", "route.json"] if not hecho(f)]
print("falta de v1:", faltan or "nada")

In [ ]:
# 4. reporte
!python /content/moi_zones.py report --res-dir {RES} --out {RES}/{OUT}
a_drive(OUT)
from google.colab import files
files.download(f"{RES}/{OUT}")

In [ ]:
# Apagar el runtime al terminar (ahorra unidades de cómputo). No depende de las celdas de arriba:
# funciona aunque el runtime se haya reiniciado. Solo apaga si el resultado completo ya está en Google Drive.
AUTO_APAGAR = True
import os, json, time
from google.colab import drive
if not os.path.exists("/content/drive/MyDrive"):
    drive.mount("/content/drive")
p = "/content/drive/MyDrive/wsi_runs/moi_zones/results_moi_zones_v2.json"
ok = os.path.exists(p) and len(json.load(open(p)).get("variants", {})) == 7
if AUTO_APAGAR and ok:
    time.sleep(60)
    from google.colab import runtime
    runtime.unassign()
else:
    print("No apago: el resultado completo no está en Drive todavía.")